## 🎯 Learning Objectives
* Implement a complete text classification pipeline using PyTorch.
* Design and build an LSTM-based neural network for sequence modeling.
* Apply data preprocessing techniques including tokenization, vocabulary creation, and padding for text data.
* Train and evaluate a deep learning model, understanding key metrics like accuracy and F1-score.


## Exercise: Build a Text Classifier with an LSTM

### Task Description

In this exercise, you will implement a complete text classification system using a Long Short-Term Memory (LSTM) network in PyTorch. Your goal is to classify short text snippets into predefined categories (e.g., positive/negative sentiment). This exercise will solidify your understanding of sequence modeling, data preprocessing for NLP, and the PyTorch training pipeline.

### Requirements

1.  **Data Preparation**: You will be provided with a mock dataset. Your solution should include:
    *   A tokenizer function.
    *   A vocabulary builder that maps tokens to unique integers.
    *   A custom PyTorch `Dataset` and `DataLoader` to handle batching and padding of sequences.
2.  **Model Implementation**: Define a PyTorch `nn.Module` for an LSTM-based text classifier. This model should include:
    *   An `nn.Embedding` layer.
    *   An `nn.LSTM` layer.
    *   A final `nn.Linear` layer for classification.
3.  **Training Loop**: Implement a standard PyTorch training loop, including:
    *   Forward pass.
    *   Loss calculation (e.g., `nn.CrossEntropyLoss`).
    *   Backward pass and optimizer step (e.g., `torch.optim.Adam`).
    *   Reporting training loss and accuracy per epoch.
4.  **Evaluation Loop**: Implement an evaluation loop to assess the model's performance on a test set, reporting:
    *   Test loss.
    *   Test accuracy.
    *   F1-score (macro average) using `sklearn.metrics`.

### Evaluation Criteria

Your solution will be evaluated based on:
*   **Correctness**: The code runs without errors and produces meaningful results.
*   **Clarity**: The code is well-structured, readable, and includes comments where necessary.
*   **Efficiency**: Reasonable training times and memory usage.
*   **Performance**: The model achieves a reasonable accuracy and F1-score on the provided mock dataset.
*   **PyTorch Idioms**: Proper use of PyTorch tensors, modules, and data utilities.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import random
import collections

# --- 2026 Ready Device Setup ---
# Prioritize Apple Silicon's MPS, then CUDA, then CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS device.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA CUDA device.")
else:
    device = torch.device("cpu")
    print("Using CPU device.")

# --- Hyperparameters ---
VOCAB_SIZE = None # Will be determined after building vocab
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 2 # For binary classification (e.g., positive/negative)
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
PAD_IDX = 0 # Index for padding token
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
N_EPOCHS = 10
MAX_SEQ_LEN = 20 # Maximum sequence length for padding

# --- Mock Dataset Generation ---
def generate_mock_dataset(num_samples=1000):
    positive_words = ["great", "amazing", "fantastic", "love", "excellent", "happy", "joy", "wonderful"]
    negative_words = ["terrible", "awful", "hate", "bad", "horrible", "sad", "disappointing", "frustrating"]
    neutral_words = ["the", "a", "is", "was", "it", "this", "that", "and", "but", "movie", "film", "story"]

    data = []
    for _ in range(num_samples // 2):
        # Positive review
        text = []
        for _ in range(random.randint(3, 10)):
            text.append(random.choice(positive_words + neutral_words))
        data.append((" ".join(text), 1))

        # Negative review
        text = []
        for _ in range(random.randint(3, 10)):
            text.append(random.choice(negative_words + neutral_words))
        data.append((" ".join(text), 0))

    random.shuffle(data)
    return data

# Generate dataset
mock_data = generate_mock_dataset(num_samples=2000)

# Split data into training and testing
train_data = mock_data[:int(len(mock_data) * 0.8)]
test_data = mock_data[int(len(mock_data) * 0.8):]

print(f"Generated {len(train_data)} training samples and {len(test_data)} test samples.")

# --- Data Preprocessing Utilities ---
def tokenize_text(text):
    return text.lower().split() # Simple space-based tokenizer

class Vocabulary:
    def __init__(self):
        self.stoi = {"<pad>": PAD_IDX, "<unk>": 1} # String to index
        self.itos = {PAD_IDX: "<pad>", 1: "<unk>"} # Index to string
        self.idx = 2

    def build_vocabulary(self, sentences):
        counts = collections.Counter()
        for sentence in sentences:
            for word in tokenize_text(sentence):
                counts[word] += 1

        # Add words that appear at least twice (simple heuristic)
        for word, count in counts.items():
            if count >= 2:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1
        global VOCAB_SIZE
        VOCAB_SIZE = len(self.stoi)
        print(f"Vocabulary built with {VOCAB_SIZE} unique tokens.")

    def numericalize(self, text):
        tokenized_text = tokenize_text(text)
        return [self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text]

# Build vocabulary from training data
vocabulary = Vocabulary()
vocabulary.build_vocabulary([text for text, label in train_data])

# --- Custom Dataset Class ---
class TextDataset(Dataset):
    def __init__(self, data, vocabulary, max_seq_len):
        self.data = data
        self.vocabulary = vocabulary
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        numericalized_text = self.vocabulary.numericalize(text)

        # Pad or truncate sequence
        if len(numericalized_text) < self.max_seq_len:
            padded_text = numericalized_text + [PAD_IDX] * (self.max_seq_len - len(numericalized_text))
        else:
            padded_text = numericalized_text[:self.max_seq_len]

        return torch.tensor(padded_text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Create datasets and dataloaders
train_dataset = TextDataset(train_data, vocabulary, MAX_SEQ_LEN)
test_dataset = TextDataset(test_data, vocabulary, MAX_SEQ_LEN)

train_iterator = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_iterator = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Example batch from train_iterator: ")
for batch_text, batch_labels in train_iterator:
    print(f"  Text batch shape: {batch_text.shape}") # Should be [BATCH_SIZE, MAX_SEQ_LEN]
    print(f"  Labels batch shape: {batch_labels.shape}") # Should be [BATCH_SIZE]
    print(f"  First text sequence (numericalized): {batch_text[0].tolist()}")
    print(f"  First label: {batch_labels[0].item()}")
    break


### Your Implementation Here

Now it's your turn! Implement the `LSTMTextClassifier` model, the training function, and the evaluation function. Then, run the training and evaluation process.

Follow these steps:

1.  **Define the `LSTMTextClassifier` class**: Inherit from `nn.Module` and implement the `__init__` and `forward` methods. Remember to handle bidirectional LSTM output if `BIDIRECTIONAL` is True.
2.  **Implement `train_model` function**: This function should take the model, iterator, optimizer, and criterion as input, perform one epoch of training, and return the average loss and accuracy.
3.  **Implement `evaluate_model` function**: This function should take the model, iterator, and criterion, perform evaluation on the test set, and return the average loss, accuracy, and F1-score.
4.  **Instantiate the model, optimizer, and loss function**: Use the hyperparameters defined in the setup cell.
5.  **Run the training loop**: Iterate for `N_EPOCHS`, calling `train_model` and `evaluate_model` for each epoch. Print the results.


In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score

# --- Model Definition ---
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout, pad_idx):
        super().__init__()

        # Embedding layer: converts integer indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # LSTM layer: processes the sequence of embeddings
        # batch_first=True means input/output tensors are (batch_size, seq_len, features)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0, # Dropout only applied between LSTM layers
            batch_first=True
        )

        # Linear layer for classification
        # If bidirectional, hidden_dim is effectively doubled for the final output
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        # Dropout layer for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text = [batch size, seq len]

        # Pass text through embedding layer
        embedded = self.dropout(self.embedding(text))
        # embedded = [batch size, seq len, embedding dim]

        # Pass embedded sequence through LSTM
        # output = [batch size, seq len, hidden_dim * num_directions]
        # hidden = [num_layers * num_directions, batch size, hidden_dim]
        # cell = [num_layers * num_directions, batch size, hidden_dim]
        output, (hidden, cell) = self.lstm(embedded)

        # We are interested in the final hidden state for classification.
        # For a bidirectional LSTM, the final hidden state `hidden` contains
        # the last forward hidden state and the last backward hidden state.
        # We concatenate these to get a single representation for the sequence.
        # hidden = [num_layers * num_directions, batch size, hidden_dim]

        # Take the last hidden state from the forward and backward directions
        # and concatenate them. If not bidirectional, it's just the last hidden state.
        if self.lstm.bidirectional:
            # hidden[-2, :, :] is the last forward hidden state
            # hidden[-1, :, :] is the last backward hidden state
            hidden = self.dropout(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1))
        else:
            # hidden[-1, :, :] is the last hidden state
            hidden = self.dropout(hidden[-1, :, :])

        # hidden = [batch size, hidden_dim * num_directions]

        # Pass through the final linear layer
        return self.fc(hidden)

# --- Training Function ---
def train_model(model, iterator, optimizer, criterion, device):
    epoch_loss = 0
    epoch_acc = 0
    model.train() # Set model to training mode

    for batch_text, batch_labels in iterator:
        batch_text = batch_text.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad() # Clear gradients from previous step

        predictions = model(batch_text)
        loss = criterion(predictions, batch_labels)

        loss.backward() # Backpropagation
        optimizer.step() # Update model parameters

        epoch_loss += loss.item()

        # Calculate accuracy
        _, predicted_labels = torch.max(predictions, 1)
        epoch_acc += (predicted_labels == batch_labels).float().sum().item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator.dataset)

# --- Evaluation Function ---
def evaluate_model(model, iterator, criterion, device):
    epoch_loss = 0
    all_preds = []
    all_labels = []
    model.eval() # Set model to evaluation mode (disables dropout, batchnorm updates)

    with torch.no_grad(): # Disable gradient calculations
        for batch_text, batch_labels in iterator:
            batch_text = batch_text.to(device)
            batch_labels = batch_labels.to(device)

            predictions = model(batch_text)
            loss = criterion(predictions, batch_labels)

            epoch_loss += loss.item()

            _, predicted_labels = torch.max(predictions, 1)
            all_preds.extend(predicted_labels.cpu().tolist())
            all_labels.extend(batch_labels.cpu().tolist())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro') # Macro average for multi-class, or binary for 2 classes

    return epoch_loss / len(iterator), accuracy, f1

# --- Model Instantiation, Optimizer, and Loss Function ---
model = LSTMTextClassifier(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    N_LAYERS,
    BIDIRECTIONAL,
    DROPOUT,
    PAD_IDX
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss().to(device)

print(f"Model architecture:\n{model}")
print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# --- Training Loop Execution ---
print("\nStarting training...")
for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_model(model, train_iterator, optimizer, criterion, device)
    test_loss, test_acc, test_f1 = evaluate_model(model, test_iterator, criterion, device)

    print(f"Epoch: {epoch+1:02}")
    print(f"\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%")
    print(f"\tTest Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% | Test F1: {test_f1:.3f}")

print("\nTraining complete!")
